# Módulo 1 · Etapa 1 — Carga e integridad de metadatos
**TFM · HAM10000** · Responsable: **Adrián Arias** · Experimento MLflow: `m1-datos`

**Objetivo:** saber exactamente qué contiene el dataset antes de analizarlo y detectar cualquier inconsistencia.

**Qué comprueba**
1. Columnas y número de filas de los metadatos.
2. Que no haya `image_id` repetidos ni filas duplicadas.
3. Que cada fila tenga su imagen y que no sobren imágenes.
4. Que cada lesión (`lesion_id`) tenga **un único diagnóstico** en todas sus fotos.
5. Recuento de imágenes y lesiones únicas.
6. Valores faltantes: edades vacías, sexo y localización `unknown`.
7. Tamaño, formato y canales de **las 10.015 imágenes**, y que ninguna esté corrupta (se decodifican todas).

Entorno: **CPU**. Secretos necesarios: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

## 1. Preparación de la sesión

In [ ]:
AUTOR = "adrian"
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"   # huella registrada en la Etapa 0

In [ ]:
import os, time, shutil, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# Copia el dataset de Drive al disco local (solo si no está ya en esta sesión).
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")

IMAGES = Path(cfg["paths"]["images_dir"])
META = Path(cfg["paths"]["metadata_csv"])
sha = file_sha256(META)
print("dataset_sha256:", sha)
assert sha == DATASET_SHA256_ESPERADO, "[!] El CSV de metadatos no es el mismo que en la Etapa 0"
print("[+] Es el mismo dataset que en la Etapa 0")

OUT = DRIVE_ROOT / "results" / "m1" / "etapa1"
OUT.mkdir(parents=True, exist_ok=True)

## 2. Funciones de comprobación

In [ ]:
# --- Lógica de la Etapa 1 ---
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
COLUMNAS = ["lesion_id", "image_id", "dx", "dx_type", "age", "sex", "localization"]

def leer_imagen(ruta):
    """Abre una imagen, la decodifica entera y devuelve sus propiedades (o el error)."""
    try:
        with Image.open(ruta) as im:
            formato, ancho, alto, modo = im.format, im.size[0], im.size[1], im.mode
            im.load()                      # decodifica todos los píxeles: detecta archivos truncados
        return dict(image_id=Path(ruta).stem, formato=formato, ancho=ancho, alto=alto,
                    modo=modo, error="")
    except Exception as e:
        return dict(image_id=Path(ruta).stem, formato=None, ancho=None, alto=None,
                    modo=None, error=f"{type(e).__name__}: {e}")

def revisar_imagenes(images_dir, workers=8):
    rutas = sorted(Path(images_dir).glob("*.jpg"))
    with ThreadPoolExecutor(workers) as ex:
        df = pd.DataFrame(list(ex.map(leer_imagen, rutas)))
    return df.astype({"ancho": "Int64", "alto": "Int64"})

def faltantes(meta):
    """Faltantes por columna: vacíos (NaN) y el valor 'unknown' que usa HAM10000."""
    filas = []
    for c in meta.columns:
        n_nan = int(meta[c].isna().sum())
        n_unk = int((meta[c].astype(str).str.lower() == "unknown").sum())
        filas.append(dict(columna=c, vacios=n_nan, unknown=n_unk,
                          total=n_nan + n_unk, pct=round(100 * (n_nan + n_unk) / len(meta), 2)))
    return pd.DataFrame(filas)

def comprobar(meta, img_info):
    """Devuelve la tabla de comprobaciones y un diccionario con los detalles de cada incidencia."""
    filas, detalle = [], {}
    def add(nombre, ok, valor, esperado):
        filas.append(dict(comprobacion=nombre, resultado="correcto" if ok else "INCIDENCIA",
                          valor=str(valor), esperado=str(esperado)))

    add("Columnas esperadas", list(meta.columns) == COLUMNAS, list(meta.columns), COLUMNAS)
    add("Filas de metadatos", len(meta) == 10015, len(meta), 10015)

    dup_img = meta[meta.image_id.duplicated(keep=False)]
    detalle["image_id_repetidos"] = dup_img
    add("image_id repetidos", len(dup_img) == 0, dup_img.image_id.nunique(), 0)

    dup_filas = int(meta.duplicated().sum())
    add("Filas completamente duplicadas", dup_filas == 0, dup_filas, 0)

    en_disco, en_meta = set(img_info.image_id), set(meta.image_id)
    sin_imagen, sin_fila = sorted(en_meta - en_disco), sorted(en_disco - en_meta)
    detalle["filas_sin_imagen"], detalle["imagenes_sin_fila"] = sin_imagen, sin_fila
    add("Filas sin imagen", len(sin_imagen) == 0, len(sin_imagen), 0)
    add("Imágenes sin fila", len(sin_fila) == 0, len(sin_fila), 0)

    n_dx = meta.groupby("lesion_id").dx.nunique()
    incoherentes = meta[meta.lesion_id.isin(n_dx[n_dx > 1].index)].sort_values("lesion_id")
    detalle["lesiones_con_varios_dx"] = incoherentes
    add("Lesiones con más de un diagnóstico", (n_dx > 1).sum() == 0, int((n_dx > 1).sum()), 0)

    clases_raras = sorted(set(meta.dx) - set(CLASES))
    add("Diagnósticos fuera de las 7 clases", not clases_raras, clases_raras or "ninguno", "ninguno")

    add("Lesiones únicas", True, meta.lesion_id.nunique(), "≈ 7.470")

    errores = img_info[img_info.error != ""]
    detalle["imagenes_corruptas"] = errores
    add("Imágenes que no se abren o están truncadas", len(errores) == 0, len(errores), 0)

    ok_img = img_info[img_info.error == ""]
    tam = ok_img.groupby(["ancho", "alto"]).size()
    detalle["tamanos"] = tam
    no_600 = int(len(ok_img) - tam.get((600, 450), 0))
    add("Imágenes con tamaño distinto de 600×450", no_600 == 0, no_600, 0)
    no_jpeg = int((ok_img.formato != "JPEG").sum())
    add("Imágenes que no son JPEG", no_jpeg == 0, no_jpeg, 0)
    no_rgb = int((ok_img.modo != "RGB").sum())
    add("Imágenes que no son RGB (3 canales)", no_rgb == 0, no_rgb, 0)
    return pd.DataFrame(filas), detalle

## 3. Carga de los metadatos

In [ ]:
meta = pd.read_csv(META)
print("Filas:", len(meta), "| Columnas:", list(meta.columns), "\n")
print(meta.dtypes.to_string(), "\n")
meta.head()

In [ ]:
# Valores que aparecen en cada columna categórica (para detectar etiquetas raras o mal escritas).
for col in ["dx", "dx_type", "sex", "localization"]:
    print(f"--- {col} ({meta[col].nunique()} valores)")
    print(meta[col].value_counts(dropna=False).to_string(), "\n")
print("--- age")
print(meta["age"].describe().round(1).to_string())
print("Valores distintos de edad:", sorted(meta["age"].dropna().unique().tolist()))

## 4. Revisión de las 10.015 imágenes

In [ ]:
t0 = time.time()
img_info = revisar_imagenes(IMAGES, workers=8)
print(f"{len(img_info)} imágenes revisadas en {time.time()-t0:.0f} s")
img_info.to_csv(OUT / "propiedades_imagenes.csv", index=False)

## 5. Comprobaciones

In [ ]:
tabla, detalle = comprobar(meta, img_info)
tabla.to_csv(OUT / "comprobaciones.csv", index=False)

def color(v):
    return "background-color: #fde2e2" if v == "INCIDENCIA" else "background-color: #e3f5e1"
display(tabla.style.map(color, subset=["resultado"]).hide(axis="index"))
print("Tamaños encontrados (ancho, alto) -> nº de imágenes:")
print(detalle["tamanos"].to_string())

In [ ]:
# Detalle de cada incidencia (si todo es "correcto", esta celda no muestra tablas).
for clave in ["image_id_repetidos", "lesiones_con_varios_dx", "imagenes_corruptas"]:
    d = detalle[clave]
    if len(d):
        print(f"[!] {clave}: {len(d)} filas"); display(d.head(20))
        d.to_csv(OUT / f"incidencia_{clave}.csv", index=False)
for clave in ["filas_sin_imagen", "imagenes_sin_fila"]:
    if detalle[clave]:
        print(f"[!] {clave}: {detalle[clave][:20]}")
n_inc = (tabla.resultado == "INCIDENCIA").sum()
print("[+] Ninguna incidencia" if n_inc == 0 else f"[!] {n_inc} comprobaciones con incidencia")

## 6. Valores faltantes
**Nota**:HAM10000 marca los faltantes de dos formas: celdas vacías (edad) y el texto `unknown` (sexo, localización).

In [ ]:
falt = faltantes(meta)
falt.to_csv(OUT / "faltantes.csv", index=False)
print(falt[falt.total > 0].to_string(index=False))

# ¿Los faltantes se concentran en alguna clase?
marca = meta.assign(edad_vacia=meta.age.isna(), sexo_unknown=meta.sex.eq("unknown"),
                    loc_unknown=meta.localization.eq("unknown"))
print("\nFaltantes por clase:")
print(marca.groupby("dx")[["edad_vacia", "sexo_unknown", "loc_unknown"]].sum().reindex(CLASES).to_string())

**Nota:** A los valores faltantes se los va a dejar porque no afectan al entrenamiento de las redes.

## 7. Tabla resumen del estado del dataset

In [ ]:
img_por_lesion = meta.groupby("lesion_id").size()
resumen = pd.DataFrame([
    ("Imágenes", len(meta)),
    ("Lesiones únicas", meta.lesion_id.nunique()),
    ("Lesiones con una sola imagen", int((img_por_lesion == 1).sum())),
    ("Lesiones con varias imágenes", int((img_por_lesion > 1).sum())),
    ("Máximo de imágenes de una lesión", int(img_por_lesion.max())),
    ("Edad vacía", int(meta.age.isna().sum())),
    ("Sexo 'unknown'", int(meta.sex.eq("unknown").sum())),
    ("Localización 'unknown'", int(meta.localization.eq("unknown").sum())),
    ("Imágenes corruptas", int((img_info.error != "").sum())),
    ("Comprobaciones con incidencia", int(n_inc)),
], columns=["Elemento", "Valor"])
resumen.to_csv(OUT / "resumen_dataset.csv", index=False)
print(resumen.to_string(index=False))
print("\nArchivos guardados en", OUT, ":", sorted(p.name for p in OUT.iterdir()))

## 8. Registro en MLflow
Run `m1-integridad` en el experimento `m1-datos`

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg)
mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])

tags = standard_tags(cfg, modulo="m1", fase="integridad", autor=AUTOR, dataset_sha256=sha)
with mlflow.start_run(run_name="m1-integridad", tags=tags) as run:
    mlflow.log_metrics({
        "n_imagenes": len(meta), "n_lesiones": meta.lesion_id.nunique(),
        "n_edad_vacia": int(meta.age.isna().sum()),
        "n_sexo_unknown": int(meta.sex.eq("unknown").sum()),
        "n_localizacion_unknown": int(meta.localization.eq("unknown").sum()),
        "n_imagenes_corruptas": int((img_info.error != "").sum()),
        "n_incidencias": int(n_inc),
    })
    mlflow.log_artifacts(str(OUT), artifact_path="etapa1")
print("[+] Run registrado:", run.info.run_id, "en", cfg["mlflow"]["experiments"]["m1"])